# Recommendation system for user

In this exercise, we develop a model that suggests similar products based on a given item. We utilize `Word2Vec()` to train the model and construct a recommendation system.
Data: https://www.kaggle.com/datasets/samantas2020/online-retail-xlsx

**Steps to Solve This Exercise**

1. Data preprocessing (Download, Handle Missing Value and Split dataset (train set: 90%,test set:10%))
2. Build a representation model for products in the training dataset. (Model, Vocabulary and Training)
3. Visualize word2vec Embeddings
4. Build a function to compute the similarity of products.

This result is based on the vector of a single product. What happens if we want to recommend products to a customer based on the products they have previously chosen?

In [13]:
# Import libraries
import os, shutil
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

## Load dataset

In [6]:
import kagglehub

data_dir = "./data"

# Download latest version
path = kagglehub.dataset_download("samantas2020/online-retail-xlsx", )
for filename in os.listdir(path):
    src = os.path.join(path, filename)
    dst = os.path.join(data_dir, filename)
    shutil.copy(src, dst)
    
print("Path to dataset files:", data_dir)

file_path = os.path.join(data_dir, "Online Retail.xlsx")
if os.path.exists(file_path):
    print("File path is:", file_path)
else:
    print("Downloading dataset is not successfully")    

Path to dataset files: ./data
File path is: ./data/Online Retail.xlsx


### Load dataset into DF

In [9]:
df = pd.read_excel(file_path)
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [11]:
print("Original shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

Original shape: (541909, 8)

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## Preprocessing data

In [ ]:
def preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    print(f"Original shape: {df.shape}\n")
    print("Missing values:\n", df.isnull().sum())

    df['Description'] = df['Description'].fillna("UNKNOWN PRODUCT")
    df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
    # df = df.dropna(subset=['Description'])

    df['Description'] = (
    df['Description']
      .str.strip()
      .str.upper()
      .str.replace(r'[^A-Z0-9\s]', '', regex=True)
      .str.replace(r'\s+', ' ', regex=True)
    )

    df = df.reset_index(drop=True)
    print(f"\n✅ Cleaned shape: {df.shape}")
    return df

In [17]:
df_processed = preprocessing(df)

Original shape: (541909, 8)

Missing values:
 InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


/tmp/ipykernel_10227/437307358.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Description'] = (



✅ Cleaned shape: (531285, 8)


In [18]:
def prepare_transactions(df):

    transactions = df.groupby('InvoiceNo')['Description'].apply(list).tolist()
    # transactions = df.groupby('InvoiceNo')['StockCode'].apply(list).tolist()
    # transactions = [t for t in transactions if len(t) >= 2]
    
    print(f"🧾 Total transactions: {len(transactions)}")
    print("🔹 Example transaction:", transactions[0][:10])
    return transactions

In [20]:
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

In [21]:
transactions = prepare_transactions(train_df)

🧾 Total transactions: 25295
🔹 Example transaction: ['WHITE HANGING HEART T-LIGHT HOLDER', 'RED WOOLLY HOTTIE WHITE HEART.', 'WHITE METAL LANTERN', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'SET 7 BABUSHKA NESTING BOXES']


## Train model and evaluation